In [1]:
import sys
import math

# Please do not remove package declarations because these are used by the autograder.

# Insert your viterbi_decoding function here, along with any subroutines you need
def viterbi_decoding(x: str,
                    alphabet: list[str],
                    states: list[str],
                    transition: list[list[float]],
                    emission: list[list[float]]) -> list[str]:
    """
    Return a path as a list of state symbols maximizing Pr(x, pi).
    Assumes uniform initial distribution over states.
    transition: n_states x n_states aligned with `states`
    emission:   n_states x n_alpha  aligned with `states` rows and `alphabet` cols
    """
    
    NEG_INF = float('-inf')
    K = len(states)
    n = len(x)
    sym = {c: i for i, c in enumerate(alphabet)}   
    # print(sym)

    
    initial_probs = math.log(1.0 / K)
    dp  = [initial_probs + (math.log(emission[k][sym[x[0]]]) if emission[k][sym[x[0]]] > 0 else NEG_INF)
           for k in range(K)] #just the first one
    # print(dp)
    ptr = [[] for _ in range(K)]   # ptr[k][i] is the best predecessor index at step i+1 prt[0] is for A and prt[1] is for B. 2 d list of current scores

    for i in range(1, n):
        si = sym[x[i]]  
        new_dp = [NEG_INF] * K
        for k in range(K):
            log_em = math.log(emission[k][si]) if emission[k][si] > 0 else NEG_INF
            best_val, best_prev = NEG_INF, 0
            for l in range(K):
                t = transition[l][k]
                val = dp[l] + (math.log(t) if t > 0 else NEG_INF) + log_em
                if val > best_val:
                    best_val, best_prev = val, l
            new_dp[k] = best_val #this is doing the max thing
            ptr[k].append(best_prev)  # add the best predecessor 
        dp = new_dp
    # print(ptr,dp)
    
    # Backtrack
    path = [max(range(K), key=lambda k: dp[k])]
    # print(path)
    for i in range(n - 2, -1, -1):
        path.append(ptr[path[-1]][i])
    path.reverse()

    #now path is [0,0,0,1,0] we need to make it A,B,A,...
    str_path=[]
    for k in path:
        str_path.append(states[k])

    return "".join(str_path)




In [ ]:
# Code Challenge: Implement the Viterbi algorithm solving the Decoding Problem.

# Input: A string x, followed by the alphabet from which x was constructed, followed by the states States, transition matrix Transition, and emission matrix Emission of an HMM (Σ, States, Transition, Emission).
# Output: A path that maximizes the (unconditional) probability Pr(x, π) over all possible paths π.
# Note: You may assume that transitions from the initial state occur with equal probability.

# Debug Datasets

# Sample Input 1:

# xyxzz
# --------
# x y z
# --------
# A B
# --------
# 	A	B
# A	0.641	0.359
# B	0.729	0.271
# --------
# 	x	y	z
# A	0.117	0.691	0.192	
# B	0.097	0.42	0.483
# Sample Output 1:

# AAABA
# Sample Input 2:

# xyxy
# --------
# x y
# --------
# A B
# --------
# 	A	B
# A	0.5	0.5	
# B	0.5	0.5
# --------
# 	x	y
# A	0.1	0.9	
# B	0.9	0.1
# Sample Output 2:

# BABA
# Sample Input 3:

# xyxy
# --------
# x y
# --------
# A B
# --------
# 	A	B
# A	0.9	0.1	
# B	0.1	0.9
# --------
# 	x	y
# A	0.5	0.5	
# B	0.5	0.5
# Sample Output 3:

# AAAA